In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler

df = pd.read_csv('preprocess_1_new.csv')

NEED_TAGS = [col for col in df.columns if col.endswith('_Need')]

In [14]:
df['Brand'].unique()

<ArrowStringArray>
[  'apple',   'honor',  'huawei',    'itel',   'meizu',   'nubia',    'oppo',
  'realme', 'samsung',   'tecno',  'xiaomi',     'zte']
Length: 12, dtype: str

In [15]:
df.columns

Index(['Unnamed: 0', 'Name', 'Screen Size', 'Display', 'NFC', 'Battery',
       'Price', 'Brand', 'OS_Name', 'OS_Version', 'antutu_11', 'clock',
       'Chipset_name', 'Chipset_gen', 'gpu_name', 'gpu_gen', 'Refresh Rate',
       'total_cores', 'max_freq_ghz', 'min_freq_ghz', 'weighted_mean',
       'rear_count', 'rear_mp_max', 'rear_f/', 'rear_ois', 'rear_telephoto',
       'rear_wide', 'front_mp', 'front_f/', 'PPI', 'SIM_total', 'has_eSIM',
       'Price_source', 'RAM_min', 'RAM_max', 'ROM_min', 'ROM_max'],
      dtype='str')

### ĐỊNH NGHĨA CÁC QUY TẮC NHÃN NHU CẦU (NEED-TAGS)

In [16]:
# Khởi tạo hoặc cập nhật trực tiếp các nhãn dựa trên quy tắc của bạn
df['Gaming_Need']        = ((df['antutu_11'] > 900_000) & (df['Refresh Rate'] >= 120)).astype(int)
df['Battery_Need']       = (df['Battery'] >= 5500).astype(int)
df['Photography_Need']   = ((df['rear_ois'] == 1) & (df['rear_count'] >= 2)).astype(int)
df['Performance_Need']   = (df['antutu_11'] > 1_200_000).astype(int)
df['Large_Display_Need'] = (df['Screen Size'] >= 6.8).astype(int)
df['HighRes_Need']       = (df['PPI'] >= 400).astype(int)
df['Multitask_Need']     = (df['RAM_min'] >= 12).astype(int)
df['Budget_King_Need']     = (df['Price'] < 4_000_000).astype(int)
df['Midrange_Value_Need']  = ((df['Price'] >= 4_000_000) & (df['Price'] <= 12_000_000)).astype(int)
df['Premium_Luxury_Need']  = (df['Price'] > 12_000_000).astype(int)

# Tự động quét và lập danh sách tất cả các cột nhãn nhu cầu hiện có
NEED_TAGS = [c for c in df.columns if c.endswith('_Need')]

# Trích xuất ma trận bộ lọc độc lập để lưu trữ riêng phục vụ Hard-Filter trên UI
tag_matrix = df[NEED_TAGS].copy()

print("=== Phân phối các Nhãn Nhu cầu sau khi khởi tạo ===")
print(df[NEED_TAGS].sum().sort_values(ascending=False))
print("-" * 50)

=== Phân phối các Nhãn Nhu cầu sau khi khởi tạo ===
Midrange_Value_Need    215
Premium_Luxury_Need    159
Gaming_Need            143
HighRes_Need           142
Performance_Need       125
Multitask_Need         114
Battery_Need            91
Budget_King_Need        71
Large_Display_Need      66
Photography_Need        52
dtype: int64
--------------------------------------------------


### FEATURE SELECTION (LOẠI BỎ BIẾN NHIỄU & CỘNG TUYẾN)

In [ ]:
cols_to_drop = [
    'clock', 'weighted_mean', 'RAM_max', 'ROM_max', 
    'front_f/', 'rear_f/', 'max_freq_ghz', 'min_freq_ghz', 
    'total_cores', 'OS_Version', 'Unnamed: 0', 'gpu_name'
]
# Đảm bảo chỉ xóa các cột nếu chúng tồn tại trong dataframe
df_cleaned = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Tách riêng mảng định danh tên máy để map kết quả khuyến nghị
phone_names = df[['Name', 'Brand', 'Price', 'Battery', 'RAM_min', 'ROM_min', 'RAM_max', 'ROM_max']].copy()

### NHÁNH CATEGORICAL PIPELINE: DOMAIN-DRIVEN GROUPING

In [18]:
top_brands = ['samsung', 'apple', 'xiaomi', 'oppo', 'honor', 'realme']
df_cleaned['Brand_Raw'] = df_cleaned['Brand'].apply(lambda x: x if x in top_brands else 'Other_Brand')

def group_chipset_tier(row):
    chipset = str(row.get("Chipset_name", "")).lower()

    # 1. Nhóm Siêu hiệu năng (Flagship)
    if "apple" in chipset:
        return "Flagship_Hardware"

    # 2. Nhóm Hiệu năng Trung bình - Khá (High-Mid)
    elif "snapdragon" in chipset or "exynos" in chipset or "kirin" in chipset:
        return "High_Mid_Hardware"

    # 3. Nhóm Hiệu năng Phổ thông / Tiết kiệm (Budget)
    else:
        return "Budget_Hardware"

# Áp dụng hàm này vào Pipeline theo từng dòng (axis=1)
df_cleaned['Hardware_Tier'] = df_cleaned.apply(group_chipset_tier, axis=1)

### TIỀN XỬ LÝ NHÁNH BIẾN SỐ (NUMERICAL PIPELINE)

In [19]:
num_features = [
    'Price', 'antutu_11', 'rear_mp_max', 'Screen Size', 
    'Battery', 'RAM_min', 'ROM_min', 'front_mp', 'Refresh Rate', 'rear_count'
]
num_features = [col for col in num_features if col in df_cleaned.columns]

# Thuần hóa các biến lệch phải nặng bằng phép biến đổi Log-Transform
log_features = ['Price', 'antutu_11', 'rear_mp_max']
for col in log_features:
    if col in df_cleaned.columns:
        df_cleaned[col] = np.log1p(df_cleaned[col])

# Áp dụng chiến lược Double-Scaler: RobustScaler trị Outliers -> MinMaxScaler đưa về [0, 1]
robust_scaler = RobustScaler()
min_max_scaler = MinMaxScaler()

numerical_matrix = robust_scaler.fit_transform(df_cleaned[num_features])
numerical_matrix = min_max_scaler.fit_transform(numerical_matrix)

df_num_scaled = pd.DataFrame(numerical_matrix, columns=num_features)

### MÃ HÓA ONE-HOT ENCODING (OHE) CHO CÁC BIẾN PHÂN LOẠI CÒN LẠI

In [20]:
cat_features = ['Brand_Raw', 'Hardware_Tier', 'OS_Name', 'Display', 'NFC', 'rear_ois', 'rear_wide', 'rear_telephoto', 'has_eSIM', 'SIM_total']
cat_features = [col for col in cat_features if col in df_cleaned.columns]

# Ép toàn bộ về kiểu chuỗi trước khi dummies để đảm bảo tính đồng bộ mã hóa
for col in cat_features:
    df_cleaned[col] = df_cleaned[col].astype(str)

df_cat_encoded = pd.get_dummies(df_cleaned[cat_features], dtype=float)

In [21]:
df_cat_encoded.shape

(445, 30)

### ĐÓNG GÓI MA TRẬN ĐẦU RA (MODEL-READY MATRICES)

In [22]:
feature_matrix = pd.concat([df_num_scaled, df_cat_encoded], axis=1)
feature_matrix = feature_matrix.clip(0, 1)

# Ma trận nhãn nhu cầu (Tag Matrix): Giữ nguyên các biến số nhị phân 0-1 để phục vụ Hard-Filter trên giao diện UI
tag_matrix = df_cleaned[NEED_TAGS].copy() if NEED_TAGS else pd.DataFrame()

In [23]:
feature_matrix.info()

<class 'pandas.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 40 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Price                            445 non-null    float64
 1   antutu_11                        445 non-null    float64
 2   rear_mp_max                      445 non-null    float64
 3   Screen Size                      445 non-null    float64
 4   Battery                          445 non-null    float64
 5   RAM_min                          445 non-null    float64
 6   ROM_min                          445 non-null    float64
 7   front_mp                         445 non-null    float64
 8   Refresh Rate                     445 non-null    float64
 9   rear_count                       445 non-null    float64
 10  Brand_Raw_Other_Brand            445 non-null    float64
 11  Brand_Raw_apple                  445 non-null    float64
 12  Brand_Raw_honor                  

### XUẤT FILE

In [24]:
feature_matrix.to_csv('feature_matrix.csv', index=False)
tag_matrix.to_csv('tag_matrix.csv', index=False)
phone_names.to_csv('phone_names.csv', index=False)

print("=== PIPELINE HOÀN THÀNH XUẤT SẮC ===")
print(f"1. Ma trận đặc trưng hình học (Tính Cosine): {feature_matrix.shape}")
print(f"2. Ma trận bộ lọc nhu cầu người dùng (UI Filter): {tag_matrix.shape}")
print(f"3. Danh mục định danh sản phẩm: {len(phone_names)} dòng máy.")

=== PIPELINE HOÀN THÀNH XUẤT SẮC ===
1. Ma trận đặc trưng hình học (Tính Cosine): (445, 40)
2. Ma trận bộ lọc nhu cầu người dùng (UI Filter): (445, 10)
3. Danh mục định danh sản phẩm: 445 dòng máy.
